# Exercise 02 — Force densities

Same net as Exercise 1, but now every bar has a force density `q = 1` (force per length, kN/m). Gravity at `p0` is 5 kN.

- What is the sum of forces in node `p0`?
- What happens if the force density `q = 2` in bars `b` and `d`?

Needs: `pip install numpy session-jupyter`

## 1. The net

In [1]:
from session_py import Color, Matrix, Point, Session
from session_jupyter import show

import numpy as np

# connectivity: rows = bars a, b, c, d   columns = nodes p0..p4
C = np.array([
    [1, -1,  0,  0,  0],
    [1,  0, -1,  0,  0],
    [1,  0,  0, -1,  0],
    [1,  0,  0,  0, -1],
])

free  = [0]            # p0
fixed = [1, 2, 3, 4]   # p1..p4

C_N = C[:, free]
C_F = C[:, fixed]

# fixed node coordinates (p1..p4)
x_F = np.array([0, 5, 0, 7])
y_F = np.array([0, 0, 7, 5])
z_F = np.array([0, 3, 3, 0])

# load on the free node: gravity 5 kN, pointing down
p_x = np.array([0])
p_y = np.array([0])
p_z = np.array([-5])

## 2. Determine p0 with q = 1

$$\mathbf{x}_N = (\mathbf{C}_N^T \mathbf{Q} \mathbf{C}_N)^{-1} (\mathbf{p}_x - \mathbf{C}_N^T \mathbf{Q} \mathbf{C}_F \mathbf{x}_F)$$

One linear solve per axis. No iterations.

In [2]:
# force densities, one per bar
q = np.array([1, 1, 1, 1])
Q = np.diag(q)

# equation 6.32
A = C_N.T @ Q @ C_N
x_N = np.linalg.solve(A, p_x - C_N.T @ Q @ C_F @ x_F)
y_N = np.linalg.solve(A, p_y - C_N.T @ Q @ C_F @ y_F)
z_N = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z_F)

print("p0 =", x_N, y_N, z_N)   # [3.] [3.] [0.25]

p0 = [3.] [3.] [0.25]


## 3. Sum of forces in node p0

Bar force = force density × length. Each bar pulls `p0` towards its fixed point.

In [3]:
# all coordinates, p0 first
x = np.concatenate([x_N, x_F])
y = np.concatenate([y_N, y_F])
z = np.concatenate([z_N, z_F])

# bar lengths and bar forces
l = np.sqrt((C @ x)**2 + (C @ y)**2 + (C @ z)**2)
f = q * l
print("bar lengths =", np.round(l, 2), "m")
print("bar forces  =", np.round(f, 2), "kN")

# pull of each bar on p0, split in x, y, z
pull_x = -q * (C @ x)
pull_y = -q * (C @ y)
pull_z = -q * (C @ z)

sum_x = pull_x.sum() + p_x[0]
sum_y = pull_y.sum() + p_y[0]
sum_z = pull_z.sum() + p_z[0]

print("sum of forces in p0 =", np.round([sum_x, sum_y, sum_z], 2), "kN")

bar lengths = [4.25 4.53 5.71 4.48] m
bar forces  = [4.25 4.53 5.71 4.48] kN
sum of forces in p0 = [0. 0. 0.] kN


The sum of forces is zero: `p0` is in equilibrium. The four bars together push up 5 kN and cancel gravity.
The bar forces are not 1 kN. They follow from the lengths: long bar, big force.

## 4. Force density q = 2 in bars b and d

In [4]:
q2 = np.array([1, 2, 1, 2])
Q2 = np.diag(q2)

A2 = C_N.T @ Q2 @ C_N
x_N2 = np.linalg.solve(A2, p_x - C_N.T @ Q2 @ C_F @ x_F)
y_N2 = np.linalg.solve(A2, p_y - C_N.T @ Q2 @ C_F @ y_F)
z_N2 = np.linalg.solve(A2, p_z - C_N.T @ Q2 @ C_F @ z_F)

x2 = np.concatenate([x_N2, x_F])
y2 = np.concatenate([y_N2, y_F])
z2 = np.concatenate([z_N2, z_F])

l2 = np.sqrt((C @ x2)**2 + (C @ y2)**2 + (C @ z2)**2)
f2 = q2 * l2

print("p0 =", x_N2, y_N2, z_N2)
print("bar lengths =", np.round(l2, 2), "m")
print("bar forces  =", np.round(f2, 2), "kN")
print("sum of forces in p0 =", np.round([p_x[0] - (q2 * (C @ x2)).sum(),
                                         p_y[0] - (q2 * (C @ y2)).sum(),
                                         p_z[0] - (q2 * (C @ z2)).sum()], 2), "kN")

p0 = [4.] [2.83333333] [0.66666667]
bar lengths = [4.95 3.8  6.23 3.76] m
bar forces  = [4.95 7.61 6.23 7.52] kN
sum of forces in p0 = [ 0. -0.  0.] kN


## 5. Look at it

`add_network` takes the connectivity matrix and the coordinates (one row `x, y, z` per node) and makes the bars and nodes.
Grey = all `q = 1`. Red = `q = 2` in bars `b` and `d`.

In [5]:
session = Session("fdm_exercise_2")

gray = Color(0.7, 0.7, 0.7, 1.0)
red = Color(0.84, 0.16, 0.16, 1.0)

session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x, y, z]).tolist()), [gray] * 4, [2.0] * 4)
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x2, y2, z2]).tolist()), [red] * 4, [3.0] * 4)

# the four fixed points, bigger
for i in fixed:
    point = Point(x[i], y[i], z[i])
    point.width = 14.0
    session.add_point(point)

show(session, "fdm_exercise_2")

`p0` moves towards `p2` and `p4`: bars `b` and `d` pull harder, so they get shorter. The node also rises: more force density carries the same load with less sag.
The sum of forces is still zero. Any choice of positive `q` gives a net in equilibrium — changing `q` is how you design the shape.